# 01 – Data Collection

This notebook collects and prepares all raw data for the project:

> **Do AI Hype Cycles Create Temporary Stock Market Bubbles?**

### Data Sources
- **Stock prices** (Yahoo Finance via `yfinance`): NVDA, MSFT, AAPL daily OHLCV
- **AI Hype – Google Trends** (`pytrends`): search interest for AI-related keywords
- **AI Hype – Reddit** (public JSON API, no key needed): daily AI mention counts from r/MachineLearning, r/artificial, r/stocks, r/investing

### Outputs (saved to `data/processed/`)
- `stock_data.csv` — daily stock prices + returns + volatility
- `trends_data.csv` — weekly Google Trends normalized hype index
- `reddit_data.csv` — daily Reddit AI mention counts
- `ai_events.csv` — key AI milestone dates
- `merged_data.csv` — merged daily panel with combined hype index and stock returns

## CELL 1: Setup – imports and directory structure

In [ ]:
import os
import time
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from pytrends.request import TrendReq

# ── Project directory setup ──────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() in ["notebook", "notebooks"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW       = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# ── Date range ───────────────────────────────────────────────────────────────
START_DATE = "2020-01-01"
END_DATE   = "2024-12-31"
TICKERS    = ["NVDA", "MSFT", "AAPL"]

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("DATA_RAW       :", DATA_RAW)
print("DATA_PROCESSED :", DATA_PROCESSED)
print(f"Date range     : {START_DATE} → {END_DATE}")

## CELL 2: Stock Data – Download NVDA, MSFT, AAPL daily prices

In [ ]:
print(f"Downloading stock data for: {TICKERS}")
raw_stock = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)

close_prices = raw_stock["Close"].copy()
close_prices.index = pd.to_datetime(close_prices.index)
close_prices.index.name = "date"

# Daily log returns
returns = np.log(close_prices / close_prices.shift(1)).dropna()
returns.columns = [f"{t}_return" for t in TICKERS]

# 20-day rolling volatility (annualised)
volatility = returns.rolling(window=20).std() * np.sqrt(252)
volatility.columns = [f"{t}_volatility" for t in TICKERS]

# Combine into one DataFrame
stock_df = close_prices.copy()
stock_df.columns = [f"{t}_close" for t in TICKERS]
stock_df = stock_df.join(returns).join(volatility)

stock_df.to_csv(DATA_PROCESSED / "stock_data.csv")
print(f"✓ Stock data saved | Shape: {stock_df.shape}")
stock_df.tail()

## CELL 3: Google Trends – AI Hype Data

In [ ]:
pytrends = TrendReq(hl="en-US", tz=360)

KEYWORDS  = ["artificial intelligence", "ChatGPT", "machine learning", "Nvidia AI"]
TIMEFRAME = f"{START_DATE} {END_DATE}"

print(f"Fetching Google Trends for: {KEYWORDS}")
pytrends.build_payload(KEYWORDS, cat=0, timeframe=TIMEFRAME, geo="", gprop="")
time.sleep(2)

trends_raw = pytrends.interest_over_time()
if "isPartial" in trends_raw.columns:
    trends_raw = trends_raw.drop(columns=["isPartial"])
trends_raw.index = pd.to_datetime(trends_raw.index)
trends_raw.index.name = "date"

# Normalize each keyword 0→1
trends_norm = trends_raw.copy().astype(float)
for col in trends_norm.columns:
    col_max = trends_norm[col].max()
    if col_max > 0:
        trends_norm[col] = trends_norm[col] / col_max

# Composite hype index
trends_norm["google_hype_index"] = trends_norm.mean(axis=1)
trends_norm.columns = ["trend_ai", "trend_chatgpt", "trend_ml", "trend_nvidia", "google_hype_index"]

trends_norm.to_csv(DATA_PROCESSED / "trends_data.csv")
print(f"✓ Google Trends saved | Shape: {trends_norm.shape}")
trends_norm.tail()

## CELL 4: Reddit Data – AI Mention Counts (No API Key Required)

We use Reddit's **public JSON search endpoint** to collect posts mentioning AI-related keywords
across key subreddits. No authentication is needed.

- Subreddits: `r/MachineLearning`, `r/artificial`, `r/stocks`, `r/investing`
- Keywords: `artificial intelligence`, `ChatGPT`, `AI`, `machine learning`
- We collect up to 100 posts per query and record their timestamps
- Posts are then aggregated into **daily mention counts**

In [ ]:
HEADERS    = {"User-Agent": "DSA210-Research/1.0 (academic project)"}
SUBREDDITS = ["MachineLearning", "artificial", "stocks", "investing"]
AI_QUERIES = ["artificial intelligence", "ChatGPT", "AI", "machine learning"]

def fetch_reddit_posts(subreddit: str, query: str, limit: int = 100) -> list:
    """
    Fetch post timestamps from a subreddit using Reddit's public JSON search API.
    Returns a list of Unix timestamps (post creation times).
    """
    url = f"https://www.reddit.com/r/{subreddit}/search.json"
    params = {
        "q": query,
        "restrict_sr": "true",
        "sort": "new",
        "limit": limit,
        "t": "all",
    }
    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=10)
        if resp.status_code == 200:
            posts = resp.json()["data"]["children"]
            return [p["data"]["created_utc"] for p in posts]
        else:
            print(f"  Warning: HTTP {resp.status_code} for r/{subreddit} | '{query}'")
            return []
    except Exception as e:
        print(f"  Error: {e}")
        return []

print("Fetching Reddit posts...")
all_timestamps = []

for subreddit in SUBREDDITS:
    for query in AI_QUERIES:
        print(f"  r/{subreddit:<20} | '{query}'")
        ts = fetch_reddit_posts(subreddit, query, limit=100)
        all_timestamps.extend(ts)
        time.sleep(1.5)  # respect Reddit rate limits

print(f"\n✓ Total posts collected: {len(all_timestamps)}")

## CELL 5: Aggregate Reddit Posts to Daily Mention Counts

In [ ]:
# Convert Unix timestamps → dates
post_dates = pd.to_datetime(all_timestamps, unit="s").normalize()

# Count posts per day
reddit_daily = (
    pd.Series(post_dates, name="date")
    .value_counts()
    .sort_index()
    .rename("reddit_ai_mentions")
    .to_frame()
)
reddit_daily.index.name = "date"

# Filter to project date range
reddit_daily = reddit_daily.loc[
    (reddit_daily.index >= START_DATE) & (reddit_daily.index <= END_DATE)
]

# Normalize 0→1
reddit_daily["reddit_hype_index"] = (
    reddit_daily["reddit_ai_mentions"] / reddit_daily["reddit_ai_mentions"].max()
)

reddit_daily.to_csv(DATA_PROCESSED / "reddit_data.csv")
print(f"✓ Reddit data saved | Shape: {reddit_daily.shape}")
print(f"Date range: {reddit_daily.index.min().date()} → {reddit_daily.index.max().date()}")
print("\nTop 10 days by AI mention count:")
display(reddit_daily.sort_values("reddit_ai_mentions", ascending=False).head(10))

## CELL 6: Define Key AI Events

In [ ]:
AI_EVENTS = {
    "GPT-3 Release":    "2020-06-11",
    "GitHub Copilot":   "2021-06-29",
    "DALL-E 2":         "2022-04-06",
    "ChatGPT Launch":   "2022-11-30",
    "GPT-4 Release":    "2023-03-14",
    "Llama 2 Release":  "2023-07-18",
    "GPT-4o Release":   "2024-05-13",
}

events_df = pd.DataFrame(list(AI_EVENTS.items()), columns=["event", "date"])
events_df["date"] = pd.to_datetime(events_df["date"])
events_df.to_csv(DATA_PROCESSED / "ai_events.csv", index=False)

print("✓ AI events saved")
print(events_df.to_string(index=False))

## CELL 7: Merge All Data Sources into One Panel

- Google Trends (weekly) → forward-filled to daily
- Reddit (daily, sparse) → forward-filled for missing trading days
- **Combined hype index** = average of Google + Reddit hype indices
- **Hype period flag** = top 25% by combined hype index

In [ ]:
# Business day index for the full date range
daily_idx = pd.date_range(start=START_DATE, end=END_DATE, freq="B")
daily_idx.name = "date"

# Google Trends: weekly → daily (forward fill)
trends_daily = trends_norm.reindex(daily_idx).ffill()

# Reddit: reindex to business days, forward fill missing
reddit_reindexed = reddit_daily.reindex(daily_idx).ffill().fillna(0)

# Start with stock data
merged = stock_df.reindex(daily_idx)

# Join Google Trends
merged = merged.join(trends_daily[[
    "google_hype_index", "trend_ai", "trend_chatgpt", "trend_ml", "trend_nvidia"
]])

# Join Reddit
merged = merged.join(reddit_reindexed[["reddit_ai_mentions", "reddit_hype_index"]])

# Combined hype index
merged["hype_index"] = merged[["google_hype_index", "reddit_hype_index"]].mean(axis=1)

# Hype period flag: top 25% hype days
hype_threshold = merged["hype_index"].quantile(0.75)
merged["is_hype_period"] = (merged["hype_index"] >= hype_threshold).astype(int)

# Drop rows with no stock data
merged = merged.dropna(subset=["NVDA_close"])

merged.to_csv(DATA_PROCESSED / "merged_data.csv")

print(f"✓ Merged dataset saved")
print(f"  Shape         : {merged.shape}")
print(f"  Date range    : {merged.index.min().date()} → {merged.index.max().date()}")
print(f"  Hype threshold: {hype_threshold:.4f} (75th percentile)")
print(f"  Hype days     : {merged['is_hype_period'].sum()} / {len(merged)}")
print(f"  Columns       : {merged.columns.tolist()}")

print("\n── Summary Statistics ──────────────────────────────")
display(merged[[
    "NVDA_close", "MSFT_close", "AAPL_close",
    "google_hype_index", "reddit_hype_index", "hype_index"
]].describe().round(3))